# Week 4 Lab — Arrays and iterative solutions

**HWRS 564a · Fall 2026**

Two ideas this week, and they belong together.

The first is `numpy`: a container for numbers that lets you say
`heads * 3.28` instead of writing a loop. The second is **iteration** — solving
equations you cannot rearrange by hand, by making a guess and improving it.
Every groundwater model you run after Week 10 is doing exactly this, several
thousand times a second.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Create numpy arrays and do arithmetic on all their elements at once
2. Aggregate along a chosen axis of a 2D array, and say what `axis=` means
3. Index and slice arrays, including with a boolean mask
4. Write a bisection solver, and check it against `scipy.optimize.brentq`
5. Step a linear reservoir forward in time with explicit and implicit schemes
6. Explain why the explicit scheme blows up, and predict the time step at which
   it starts to

---

## Part 1 — Why arrays

Here is last week's unit conversion, done the week-3 way.

In [ ]:
import numpy as np

depths_m = [31.2, 45.8, 88.4, 12.0, 67.3]

depths_ft = []
for d in depths_m:
    depths_ft.append(d * 3.28084)

print(depths_ft)

And the same thing with an array:

In [ ]:
depths = np.array([31.2, 45.8, 88.4, 12.0, 67.3])

print(depths * 3.28084)

The array version is shorter, but that is the least of it. It is also **much
faster**, because the loop happens in compiled code rather than in Python.

In [ ]:
import time

big_list = list(range(1_000_000))
big_array = np.arange(1_000_000)

t0 = time.perf_counter()
_ = [x * 2.5 for x in big_list]
list_time = time.perf_counter() - t0

t0 = time.perf_counter()
_ = big_array * 2.5
array_time = time.perf_counter() - t0

print(f"list comprehension: {list_time * 1000:7.1f} ms")
print(f"numpy:              {array_time * 1000:7.1f} ms")
print(f"speedup:            {list_time / array_time:7.0f}x")

By Week 11 your model grids have hundreds of thousands of cells and you will
touch them once per iteration. A 50× difference stops being an academic point.

An array has a `shape`, a `dtype`, and a fixed size. A list has none of those.

In [ ]:
print(f"shape: {depths.shape}")
print(f"dtype: {depths.dtype}")
print(f"ndim:  {depths.ndim}")

# Everything in an array has ONE type. Mix them and numpy picks one for you:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2, 3.5]).dtype)

### YOUR TURN 1

`np.array` arithmetic is elementwise. Given the arrays below, compute the
**hydraulic head elevation** at each well — land surface elevation minus depth
to water — as an array called `head_elev`.

Then compute `head_range`, the difference between the highest and lowest head.

Do both without writing a loop.

In [ ]:
land_surface = np.array([730.0, 742.5, 715.0, 760.2, 728.8])   # m above datum
depth_to_water = np.array([31.2, 45.8, 88.4, 12.0, 67.3])      # m below surface

# YOUR TURN
head_elev = ...
head_range = ...

In [ ]:
# CHECK
assert isinstance(head_elev, np.ndarray), "head_elev should be a numpy array, not a list"
assert head_elev.shape == (5,), f"expected shape (5,), got {head_elev.shape}"
np.testing.assert_allclose(head_elev, [698.8, 696.7, 626.6, 748.2, 661.5])
assert abs(head_range - 121.6) < 1e-9, f"expected 121.6, got {head_range}"
print(f"heads: {head_elev}")
print(f"range: {head_range:.1f} m. Correct.")

---

## Part 2 — Two dimensions, and what `axis` means

A model grid is 2D, so most arrays you meet after Week 10 are too. Here is a
small head field: 4 rows (north to south) by 6 columns (west to east).

In [ ]:
heads = np.array([
    [412.1, 411.4, 410.2, 408.9, 406.3, 403.1],
    [412.0, 411.2, 409.8, 408.1, 405.7, 402.9],
    [411.8, 410.9, 409.1, 406.2, 405.1, 402.6],
    [411.7, 410.8, 409.6, 408.0, 405.4, 402.7],
])

print(f"shape: {heads.shape}  ->  {heads.shape[0]} rows, {heads.shape[1]} columns")
print(f"overall mean: {heads.mean():.2f} m")

`axis` is the dimension that gets **collapsed**, not the one you keep. That
phrasing is the only way I've found to remember it.

In [ ]:
print("axis=0 collapses the rows, leaving one value per column:")
print(np.round(heads.mean(axis=0), 2), f"  shape {heads.mean(axis=0).shape}")
print()
print("axis=1 collapses the columns, leaving one value per row:")
print(np.round(heads.mean(axis=1), 2), f"  shape {heads.mean(axis=1).shape}")

Indexing is `[row, column]`, and slices work in both positions at once.

In [ ]:
print(f"single cell  heads[2, 3]:  {heads[2, 3]}")
print(f"whole row    heads[2]:     {heads[2]}")
print(f"whole column heads[:, 3]:  {heads[:, 3]}")
print(f"top-left 2x2 heads[:2,:2]:\n{heads[:2, :2]}")

A **boolean mask** selects by condition. This is how you will pull "every cell
below the confining unit" out of a model array without a loop.

In [ ]:
mask = heads < 406.0
print(mask)
print(f"\n{mask.sum()} of {heads.size} cells are below 406 m")
print(f"their values: {heads[mask]}")

### YOUR TURN 2

The hydraulic gradient across the grid runs roughly west to east. Compute:

- `column_means` — the mean head in each **column** (one value per column)
- `steepest_row` — the **index** of the row with the largest head drop from its
  first to its last column

`np.argmax` returns the index of the largest value in an array.

In [ ]:
# YOUR TURN
column_means = ...
steepest_row = ...

In [ ]:
# CHECK
assert column_means.shape == (6,), f"expected 6 column means, got shape {column_means.shape}"
np.testing.assert_allclose(column_means, [411.9, 411.075, 409.675, 407.8, 405.625, 402.825])
assert steepest_row == 2, f"expected row 2, got {steepest_row}"
drops = heads[:, 0] - heads[:, -1]
print(f"column means: {np.round(column_means, 2)}")
print(f"head drop per row: {np.round(drops, 2)}")
print(f"steepest is row {steepest_row}. Correct.")

**Worth noticing:** row 2 has the steepest drop because it contains that 406.2
in column 3 — a local low. In a real model that would be a pumping well, and
spotting it in the numbers before you plot is a good habit.

---

## Part 3 — Solving an equation you can't rearrange

An unconfined aquifer drains to a stream. Under the Dupuit assumptions, the
discharge per unit width to the stream is

$$q(h) = \frac{K\,(h^2 - h_{stream}^2)}{2L}$$

At steady state that discharge has to equal the recharge collected upgradient,
$R \cdot L$. So the equilibrium water table height $h$ is the value where

$$f(h) = \frac{K\,(h^2 - h_{stream}^2)}{2L} - R\,L = 0$$

You *could* rearrange this one by hand — it's quadratic. Most of the equations
you meet after Week 10 you cannot, so we will solve it the way you'd solve those.

In [ ]:
K = 15.0          # m/d
L = 900.0         # m, distance from the divide to the stream
H_STREAM = 8.0    # m, saturated thickness at the stream
R = 3.5e-4        # m/d recharge


def f(h):
    """Residual: discharge to the stream minus recharge collected. Zero at equilibrium."""
    return K * (h**2 - H_STREAM**2) / (2 * L) - R * L


for h in [8.0, 10.0, 12.0, 15.0, 20.0]:
    print(f"f({h:5.1f}) = {f(h):8.3f}")

The residual changes sign between 10 and 12 m, so the root is in there. That is
the entire idea behind **bisection**: if a continuous function has opposite signs
at two points, it has a root between them. Cut the interval in half, keep the
half that still brackets the root, repeat.

In [ ]:
# One step, by hand, so the loop below isn't magic.
lo, hi = 10.0, 12.0
mid = (lo + hi) / 2
print(f"f(lo)={f(lo):.3f}  f(mid)={f(mid):.3f}  f(hi)={f(hi):.3f}")
print(f"sign changes between lo and mid, so the root is in [{lo}, {mid}]")

### YOUR TURN 3 — write the solver

Fill in `bisect`. The loop is written for you; you supply the three lines that
make it work.

At each step:

1. `mid` is the midpoint of `lo` and `hi`
2. if `func(lo)` and `func(mid)` have **opposite signs**, the root is in the
   lower half, so `hi = mid`; otherwise `lo = mid`
3. stop when `hi - lo` is smaller than `tol`

Two things to get right, both of which come back in Week 12:

- Compare with a **tolerance**, never `==`. You met this in Week 2.
- Carry a `max_iter` guard, like the `while` loop in Week 3.

In [ ]:
def bisect(func, lo, hi, tol=1e-8, max_iter=200):
    """Find a root of `func` bracketed by `lo` and `hi`.

    Returns
    -------
    (root, n_iterations, history) : the root, how many halvings it took, and
    the midpoint at every step so we can plot the convergence.
    """
    if func(lo) * func(hi) > 0:
        raise ValueError(f"f({lo}) and f({hi}) have the same sign — no bracket")

    history = []
    for n in range(1, max_iter + 1):
        # YOUR TURN
        mid = ...
        history.append(mid)

        if hi - lo < tol:
            return mid, n, history

        if func(lo) * func(mid) < 0:
            hi = ...
        else:
            lo = ...

    raise RuntimeError(f"no convergence in {max_iter} iterations")

In [ ]:
# CHECK
root, n_iter, history = bisect(f, 8.0, 20.0)
assert abs(f(root)) < 1e-6, f"f(root) should be ~0, got {f(root)}"
assert abs(root - 10.0896) < 1e-3, f"root looks wrong: {root}"
assert 25 < n_iter < 40, f"{n_iter} iterations is suspicious for tol=1e-8 on a 12 m bracket"

from scipy.optimize import brentq
reference = brentq(f, 8.0, 20.0)
assert abs(root - reference) < 1e-6, f"disagrees with scipy: {root} vs {reference}"

print(f"equilibrium saturated thickness: {root:.4f} m")
print(f"found in {n_iter} halvings")
print(f"scipy.optimize.brentq agrees:    {reference:.4f} m")
print("Correct.")

Bisection is slow but it cannot fail if you hand it a real bracket. `brentq`
mixes bisection with faster methods and gets there in about a quarter of the
steps — which is why you use the library version in real work, and why it was
worth writing your own once.

Here is what "halving" actually looks like:

In [ ]:
import matplotlib.pyplot as plt

error = np.abs(np.array(history) - reference)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.semilogy(range(1, len(error) + 1), error, "o-", color="#AB0520", markersize=4)
ax.set_xlabel("iteration")
ax.set_ylabel("|error| (m)")
ax.set_title("Bisection halves the error every step — a straight line on a log axis")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 4 — Stepping a reservoir through time

A linear reservoir drains at a rate proportional to how full it is:

$$\frac{dS}{dt} = -kS \qquad\Longrightarrow\qquad S(t) = S_0 e^{-kt}$$

We know the answer here, which is exactly why it is worth simulating: you can
check the numerical method against the truth. For the nonlinear version at the
end — and for MODFLOW — there is no truth to check against.

The **explicit** (forward Euler) scheme evaluates the rate at the step you are
leaving:

$$S_{n+1} = S_n - k S_n \Delta t$$

In [ ]:
S0 = 100.0     # initial storage
k = 10.0       # 1/day — a fast-draining reservoir
dt = 0.05      # days
n_steps = 60

t = np.arange(n_steps + 1) * dt

S_explicit = np.zeros(n_steps + 1)
S_explicit[0] = S0
for n in range(n_steps):
    S_explicit[n + 1] = S_explicit[n] - k * S_explicit[n] * dt

S_analytic = S0 * np.exp(-k * t)

print(f"after {t[-1]:.2f} days:  explicit {S_explicit[-1]:.4f}   analytic {S_analytic[-1]:.4f}")

Close enough. Now raise the time step to 0.15 days and run the identical code.

In [ ]:
dt_big = 0.15
t_big = np.arange(n_steps + 1) * dt_big

S_bad = np.zeros(n_steps + 1)
S_bad[0] = S0
for n in range(n_steps):
    S_bad[n + 1] = S_bad[n] - k * S_bad[n] * dt_big

print(S_bad[:8].round(2))

Storage goes **negative**, then swings further from zero every step. The
reservoir is emptying into a hole that gets deeper. This is not a bug in the
code; it is the scheme failing.

Look at the update rule: $S_{n+1} = S_n(1 - k\Delta t)$. Every step multiplies
by $(1 - k\Delta t)$. With $k\Delta t = 1.5$ that factor is $-0.5$ — the sign
flips — and with $k\Delta t > 2$ its magnitude exceeds 1 and the whole thing
diverges. **The explicit scheme is only stable for $\Delta t < 2/k$**, and only
non-oscillating for $\Delta t < 1/k$.

The **implicit** (backward Euler) scheme evaluates the rate at the step you are
arriving at:

$$S_{n+1} = S_n - k S_{n+1} \Delta t \qquad\Longrightarrow\qquad S_{n+1} = \frac{S_n}{1 + k \Delta t}$$

The factor $1/(1 + k\Delta t)$ is between 0 and 1 for *any* positive time step,
so it never oscillates and never diverges.

### YOUR TURN 4

Fill in the implicit update and run it at the same `dt_big` that broke the
explicit scheme.

In [ ]:
S_implicit = np.zeros(n_steps + 1)
S_implicit[0] = S0

for n in range(n_steps):
    # YOUR TURN
    S_implicit[n + 1] = ...

In [ ]:
# CHECK
assert np.all(S_implicit >= 0), "implicit storage should never go negative"
assert np.all(np.diff(S_implicit) <= 0), "storage should decrease monotonically"
np.testing.assert_allclose(S_implicit[1], 40.0, rtol=1e-9)
np.testing.assert_allclose(S_implicit[2], 16.0, rtol=1e-9)
print(f"first four steps: {S_implicit[:4].round(3)}")
print("Stable at a time step that made the explicit scheme diverge. Correct.")

Stable, but look at how *wrong* it is: the analytic answer after one step of
0.15 d is $100e^{-1.5} = 22.3$, and implicit Euler says 40.0.

That is the trade every numerical scheme makes. Explicit is accurate until it
explodes. Implicit never explodes and is always a bit damped. MODFLOW is
implicit, which is why it will happily hand you a converged answer at a time
step far too coarse to mean anything.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=False)

axes[0].plot(t, S_analytic, color="k", lw=2, label="analytic")
axes[0].plot(t, S_explicit, "o--", color="#AB0520", ms=3, label=f"explicit, dt={dt}")
axes[0].set_title(f"stable: k*dt = {k * dt:.2f}")
axes[0].legend(frameon=False)

axes[1].plot(t_big, S0 * np.exp(-k * t_big), color="k", lw=2, label="analytic")
axes[1].plot(t_big, S_bad, "o--", color="#AB0520", ms=3, label=f"explicit, dt={dt_big}")
axes[1].plot(t_big, S_implicit, "s-", color="#0C234B", ms=3, label=f"implicit, dt={dt_big}")
axes[1].set_ylim(-120, 120)
axes[1].axhline(0, color="grey", lw=0.6)
axes[1].set_title(f"unstable: k*dt = {k * dt_big:.2f}")
axes[1].legend(frameon=False, fontsize=8)

for ax in axes:
    ax.set_xlabel("time (days)")
    ax.set_ylabel("storage")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### YOUR TURN 5 — vectorize it

For the *linear* reservoir the explicit loop has a closed form: every step
multiplies by the same factor, so after $n$ steps

$$S_n = S_0 (1 - k\Delta t)^n$$

Compute `S_vectorized` for the stable case with **no Python loop**, and confirm
it matches `S_explicit` exactly.

`np.arange(n_steps + 1)` gives you the step numbers, and `**` works elementwise.

In [ ]:
# YOUR TURN
S_vectorized = ...

In [ ]:
# CHECK
assert isinstance(S_vectorized, np.ndarray)
assert S_vectorized.shape == S_explicit.shape, f"shape {S_vectorized.shape} != {S_explicit.shape}"
np.testing.assert_allclose(S_vectorized, S_explicit, rtol=1e-10)
print(f"first four: {S_vectorized[:4].round(4)}")
print("Identical to the loop, and it fits on one line. Correct.")

Two honest caveats, because "vectorize everything" is bad advice:

- This worked because the linear reservoir has a closed form. **Most time loops
  do not** — each step genuinely depends on the last, and you have to loop.
- What you *can* almost always vectorize is the work *inside* a step. In
  MODFLOW terms: you loop over time steps, but never over cells.

---

## Part 5 — Stretch: the nonlinear reservoir

Real catchments do not drain linearly. A common form is $Q = aS^b$ with
$b > 1$, so a full reservoir drains disproportionately faster than an empty one.

With inflow $P$ and loss $E$:

$$\frac{dS}{dt} = P - E - aS^b$$

There is no closed-form solution. The loop is all you have.

### YOUR TURN 6

Write the explicit time loop for the nonlinear reservoir, then look at what
changing `b` does.

In [ ]:
P, E = 50.0, 15.0     # inflow and loss, units/day
a = 0.1               # discharge coefficient
S0_nl = 100.0
dt_nl = 0.05
n_nl = 2000        # 100 days — long enough for every curve to reach steady state

t_nl = np.arange(n_nl + 1) * dt_nl


def simulate_nonlinear(b):
    S = np.zeros(n_nl + 1)
    S[0] = S0_nl
    for n in range(n_nl):
        # YOUR TURN
        discharge = ...
        S[n + 1] = ...
    return S

In [ ]:
# CHECK
S_b1 = simulate_nonlinear(1.0)
assert S_b1.shape == (n_nl + 1,)
# With b = 1 this reduces to a linear reservoir with an inflow, whose steady
# state is where a*S = P - E, i.e. S = 350.
assert abs(S_b1[-1] - 350.0) < 1.0, f"b=1 should approach S=350, got {S_b1[-1]:.1f}"

S_b15 = simulate_nonlinear(1.5)
steady_b15 = ((P - E) / a) ** (1 / 1.5)
assert abs(S_b15[-1] - steady_b15) < 1.0, f"b=1.5 should approach {steady_b15:.1f}"
print(f"b = 1.0 settles at {S_b1[-1]:7.1f}   (analytic {(P - E) / a:.1f})")
print(f"b = 1.5 settles at {S_b15[-1]:7.1f}   (analytic {steady_b15:.1f})")
print("Correct.")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for b, style in [(1.0, "-"), (1.2, "--"), (1.5, "-."), (2.0, ":")]:
    ax.plot(t_nl, simulate_nonlinear(b), style, lw=2, label=f"b = {b}")
ax.set_xlabel("time (days)")
ax.set_ylabel("storage")
ax.set_title("Nonlinear reservoir: sensitivity to the discharge exponent")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Think about this before next week:** every curve starts at 100 and every one
approaches a steady state, but they disagree by a factor of ten about where. The
exponent `b` is the least-well-known parameter in the whole model and it
dominates the answer.

That is a calibration problem, and it is the reason Project 2 exists.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

**HW 2 — Control flow and functions**, Wednesday 9/16 at 11:59pm, through D2L.

## Next week

`pandas`: the same array ideas, but with labels on the rows and columns, so you
can stop remembering that column 3 is the water level.

## Stuck?

- `ValueError: operands could not be broadcast together` means two arrays have
  shapes that don't line up. Print `.shape` on both — it is almost always a
  `(5,)` meeting a `(5, 1)`.
- An array that prints as all `nan` usually came from dividing by zero or taking
  the log of a negative number several cells earlier.
- If a `while` or `for` loop seems to hang, interrupt the kernel (■) and check
  that whatever the condition depends on actually changes inside the loop.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.